<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/main/STEP_6_Q_LEARNING_IMPLEMENTATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 6: Q-LEARNING IMPLEMENTATION**

# Dual-Goal Reward Fuction

In [ ]:
baseline_profit = train_df["Order Profit Per Order"].mean()
baseline_days = train_df["Days for shipping (real)"].replace(0, np.nan).mean()

min_profit = baseline_profit * 1.00      # at least no profit loss
max_days = baseline_days * 0.90          # at least 10% faster
min_orders = 10

action_stats = train_df.groupby("action").agg(
    Orders=("Order Id", "count"),
    Avg_Profit=("Order Profit Per Order", "mean"),
    Avg_Days=("Days for shipping (real)", "mean")
).reset_index()

baseline_profit = train_df["Order Profit Per Order"].mean()
baseline_days = train_df["Days for shipping (real)"].mean()

good_actions = action_stats[
    (action_stats["Orders"] >= min_orders) &
    (action_stats["Avg_Profit"] >= baseline_profit * 0.98) &
    (action_stats["Avg_Days"] <= baseline_days * 0.98)
]["action"]

train_df = train_df[train_df["action"].isin(good_actions)].copy()

print("Actions retained:", train_df["action"].nunique())
print("Rows retained:", len(train_df))

In [ ]:
import pandas as pd

profit_weight = 3.0
days_weight = 0.5

target_profit = train_df["Order Profit Per Order"].mean() * 1.10
target_days = train_df["Days for shipping (real)"].replace(0, np.nan).mean() * 0.80

profit_scaled = (
    train_df["Order Profit Per Order"]
    - train_df["Order Profit Per Order"].mean()
) / train_df["Order Profit Per Order"].std()

days_scaled = (
    train_df["Days for shipping (real)"]
    - train_df["Days for shipping (real)"].mean()
) / train_df["Days for shipping (real)"].std()

train_df["reward"] = (
    2.0 * profit_scaled
    - 1.0 * days_scaled
)
action_check = train_df.groupby("action").agg(
    avg_profit=("Order Profit Per Order", "mean"),
    avg_days=("Days for shipping (real)", "mean"),
    count=("action", "count")
).reset_index()

print("Target profit:", target_profit)
print("Target days:", target_days)

action_check[
    (action_check["avg_profit"] >= target_profit) &
    (action_check["avg_days"] <= target_days) &
    (action_check["count"] >= 10)
].sort_values(["avg_profit", "avg_days"], ascending=[False, True])

In [ ]:
import pandas as pd
import numpy as np

# Ensure train_df is available (it's already loaded and filtered in preceding cells)
# train_df = pd.read_csv("dataco_rl_train_discrete.csv", encoding="latin1")

# Aggregate metrics by action to calculate normalization values
action_summary = train_df.groupby("action").agg(
    Avg_Profit=("Order Profit Per Order", "mean"),
    Avg_Days=("Days for shipping (real)", "mean"),
    Avg_Route_Mode_Cost=("Route_Mode_Cost", "mean")
).reset_index()

# Define a small epsilon to prevent division by zero in normalization
EPSILON = 1e-8

# Normalize Avg_Profit (higher is better)
min_profit_val = action_summary["Avg_Profit"].min()
max_profit_val = action_summary["Avg_Profit"].max()
profit_range = max_profit_val - min_profit_val
normalized_profit = (action_summary["Avg_Profit"] - min_profit_val) / (profit_range + EPSILON)

# Normalize Avg_Days for on_time_rate (lower days are better, so inverse scaling)
min_days_val = action_summary["Avg_Days"].min()
max_days_val = action_summary["Avg_Days"].max()
days_range = max_days_val - min_days_val
on_time_rate = 1 - ((action_summary["Avg_Days"] - min_days_val) / (days_range + EPSILON))

# Normalize Avg_Route_Mode_Cost (higher cost is worse, so keep direct scaling for negative coefficient)
min_cost_val = action_summary["Avg_Route_Mode_Cost"].min()
max_cost_val = action_summary["Avg_Route_Mode_Cost"].max()
cost_range = max_cost_val - min_cost_val
normalized_cost = (action_summary["Avg_Route_Mode_Cost"] - min_cost_val) / (cost_range + EPSILON)

action_score = (
      0.6 * normalized_profit
    + 0.3 * on_time_rate
    - 0.1 * normalized_cost
)

# Initialize Q-Table and Train

In [ ]:
import joblib

# Create new LabelEncoders for the *filtered* data to ensure contiguous IDs for Q-table indexing.
# This is necessary because filtering might have removed some states/actions,
# leading to gaps in the IDs and out-of-bounds errors when indexing the Q-table.

# Fit state_le_q on the *original integer state_ids* to map them to contiguous 0-indexed values for Q-table.
# Explicitly cast to int to ensure correct fitting.
state_le_q = LabelEncoder()
train_df["q_state_id"] = state_le_q.fit_transform(train_df["state_id"].astype(int))

# Fit action_le_q on the *action strings* to enable correct inverse transformation back to strings.
# Explicitly cast to str to ensure correct fitting.
action_le_q = LabelEncoder()
train_df["q_action_id"] = action_le_q.fit_transform(train_df["action"].astype(str))

# Use these new 'q_state_id' and 'q_action_id' for Q-learning
valid_state_actions = (
    train_df.groupby("q_state_id")["q_action_id"]
    .apply(lambda x: sorted(x.unique()))
    .to_dict()
)

n_states = train_df["q_state_id"].nunique()
n_actions = train_df["q_action_id"].nunique()

Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.9
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

reward_table = (
    train_df.groupby(["q_state_id", "q_action_id"])["reward"]
    .mean()
    .to_dict()
)

state_ids = train_df["q_state_id"].to_numpy()

for episode in range(episodes):
    total_reward = 0

    for i in range(len(state_ids) - 1):
        state = int(state_ids[i])
        next_state = int(state_ids[i + 1])

        valid_actions = valid_state_actions.get(state, [])

        if len(valid_actions) == 0:
            continue

        if np.random.rand() < epsilon:
            action = np.random.choice(valid_actions)
        else:
            action = max(valid_actions, key=lambda a: Q[state, a])

        reward = reward_table.get((state, action), 0)

        next_valid_actions = valid_state_actions.get(next_state, [])

        if len(next_valid_actions) > 0:
            next_best_q = max(Q[next_state, a] for a in next_valid_actions)
        else:
            next_best_q = 0

        Q[state, action] += alpha * (
            reward + gamma * next_best_q - Q[state, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}")

print("Training complete")

# Save the encoders for use in later cells for inverse transformation.
# action_le_q is fitted on strings, state_le_q is fitted on original integer state_ids.
joblib.dump(action_le_q, "action_le_q.pkl")
joblib.dump(state_le_q, "state_le_q.pkl")

# Extract the Learned Policy

In [ ]:
import joblib

# Load the specific action encoder used for Q-learning
action_le_q = joblib.load("action_le_q.pkl")

policy_action_ids = []

for s in range(n_states):
    valid_actions = valid_state_actions.get(s, [])

    if len(valid_actions) == 0:
        # If no valid actions, choose an arbitrary action (e.g., the one with max Q-value)
        # This might need refinement depending on desired behavior for unseen states
        # If Q[s] is all zeros, argmax will return 0. This is an arbitrary choice.
        policy_action_ids.append(np.argmax(Q[s]))
    else:
        best_action = max(valid_actions, key=lambda a: Q[s, a])
        policy_action_ids.append(best_action)

policy_df = pd.DataFrame({
    "state_id": np.arange(n_states), # These are the q_state_id values
    "action_id": policy_action_ids, # These are the q_action_id values
    "recommended_action": action_le_q.inverse_transform(policy_action_ids).astype(str) # Ensure string type
})

policy_df[
    [
        "Recommended_Fulfillment_Region",
        "Recommended_Route",
        "Recommended_Shipping_Mode"
    ]
] = policy_df["recommended_action"].str.split(" \\| ", expand=True)

policy_df.head()

# Evaluate Recommended Shipping Modes

In [ ]:
import joblib
import pandas as pd # Import pandas

# Load the specific action encoder used for Q-learning
action_le_q = joblib.load("action_le_q.pkl")
state_le_q = joblib.load('state_le_q.pkl') # Load state_le_q

# Load the processed eval_df
eval_df = pd.read_csv("dataco_rl_validation_processed.csv", encoding="latin1")

# Extract best action for each state (using q_state_id and q_action_id from Q-table)
# The 'policy' array contains the q_action_id for each q_state_id
policy = np.argmax(Q, axis=1)

# Convert encoded actions back to route/shipping-mode labels using action_le_q
policy_actions = action_le_q.inverse_transform(policy).astype(str) # Ensure string type

# Create a policy DataFrame based on q_state_id and q_action_id
policy_df_local = pd.DataFrame({
    "q_state_id": np.arange(len(policy_actions)), # These are the re-encoded state IDs
    "recommended_action": policy_actions
})

# Ensure eval_df has the 'q_state_id' for merging
# It's important that `state_le_q` was fitted on the same set of original state_ids
# present in eval_df. We use the original integer state_id from eval_df for transformation.
# Filter eval_df to only include state_ids that state_le_q knows about (from filtered train_df)
known_original_state_ids = state_le_q.classes_

# Make a copy to avoid SettingWithCopyWarning
# If eval_df already processed, re-filter and re-calculate q_state_id for robustness
if 'state_id' in eval_df.columns:
    eval_df_temp = eval_df[eval_df["state_id"].isin(known_original_state_ids)].copy()
    if not eval_df_temp.empty:
        eval_df_temp["q_state_id"] = state_le_q.transform(eval_df_temp["state_id"].astype(int))
    else:
        eval_df_temp["q_state_id"] = [] # Ensure column exists even if empty
    eval_df = eval_df_temp
else:
    # Handle case where state_id might not be directly available, which would be an upstream issue
    print("Warning: 'state_id' column not found in eval_df. Ensure eval_df is properly prepared.")
    # Attempt to proceed by creating an empty q_state_id column if eval_df is also empty
    if eval_df.empty:
        eval_df["q_state_id"] = []

# Before merging, drop any existing 'recommended_action' column from eval_df
# to avoid suffix issues if this cell is re-run and eval_df is already partially processed.
if 'recommended_action' in eval_df.columns:
    eval_df = eval_df.drop(columns=['recommended_action'])

# Now merge using 'q_state_id'
eval_df = eval_df.merge(policy_df_local, on="q_state_id", how="left")

# If action contains route/mode like:
# Fulfillment_Region | Route | Shipping_Mode
# Check if 'recommended_action' exists after merge and if eval_df is not empty
if 'recommended_action' in eval_df.columns and not eval_df.empty:
    eval_df["Recommended_Shipping_Mode"] = (
        eval_df["recommended_action"]
        .astype(str) # Ensure it's string before split
        .str.split(" \\| ")
        .str[-1]
        .str.strip()
    )

    # Compare historical vs recommended shipping mode
    mode_comparison = pd.crosstab(
        eval_df["Shipping Mode_str"],
        eval_df["Recommended_Shipping_Mode"],
        normalize="index"
    ) * 100

    print("Historical vs Recommended Shipping Mode (%):")
    display(mode_comparison.round(2))

    # Recommended mode distribution
    recommended_distribution = (
        eval_df["Recommended_Shipping_Mode"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print("Recommended Shipping Mode Distribution (%):")
    display(recommended_distribution)

    # Historical mode distribution
    historical_distribution = (
        eval_df["Shipping Mode_str"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print("Historical Shipping Mode Distribution (%):")
    display(historical_distribution)
else:
    print("Eval DataFrame is empty or 'recommended_action' column is not available after merge. Cannot perform mode comparison.")

In [ ]:
recommended_mode_summary = (
    eval_df.groupby("Recommended_Shipping_Mode")
    .agg(
        orders=("Order Id", "count"),
        avg_profit=("Order Profit Per Order", "mean"),
        avg_late_risk=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_route_mode_cost=("Route_Mode_Cost", "mean"),
        avg_reward=("reward", "mean")
    )
    .sort_values("avg_reward", ascending=False)
)

display(recommended_mode_summary.round(4))

# Measure Improvement

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the Q-table specific encoders
action_le_q = joblib.load("action_le_q.pkl")
state_le_q = joblib.load("state_le_q.pkl")

# Load the processed validation dataframe
eval_df = pd.read_csv("dataco_rl_validation_processed.csv", encoding="latin1")

# Ensure eval_df only contains states and actions that the Q-table understands
# Filter by actions first (string values)
eval_df = eval_df[eval_df["action"].isin(action_le_q.classes_)].copy()

# Filter by original state_id (integer values) that state_le_q was fitted on
# state_le_q.classes_ holds the original state_id integers that were retained after filtering train_df
eval_df = eval_df[eval_df["state_id"].isin(state_le_q.classes_)].copy()

if eval_df.empty:
    print("Eval DataFrame is empty after filtering for Q-table compatible states and actions. Cannot compute Q-values.")
    # Initialize q_results with NaNs if no data to process
    q_results = pd.DataFrame({
        "Metric": ["Avg Q-Value"],
        "Historical": [np.nan],
        "Recommended Policy": [np.nan],
        "Change": [np.nan],
        "Percent Change": [np.nan]
    })
    display(q_results.round(4))
else:
    # Transform eval_df's original state_id and action strings to Q-table indices
    eval_df["q_state_id"] = state_le_q.transform(eval_df["state_id"].astype(int))
    eval_df["q_action_id"] = action_le_q.transform(eval_df["action"].astype(str))

    # Compare historical Q-value vs recommended Q-value
    # Use q_state_id and q_action_id for indexing the Q-table
    eval_df["historical_q"] = [
        Q[s, a] for s, a in zip(eval_df["q_state_id"], eval_df["q_action_id"])
    ]

    eval_df["policy_q"] = [
        Q[s].max() for s in eval_df["q_state_id"]
    ]

    q_results = pd.DataFrame({
        "Metric": ["Avg Q-Value"],
        "Historical": [eval_df["historical_q"].mean()],
        "Recommended Policy": [eval_df["policy_q"].mean()]
    })

    q_results["Change"] = (
        q_results["Recommended Policy"] - q_results["Historical"]
    )

    q_results["Percent Change"] = (
        q_results["Change"] / q_results["Historical"].replace(0, np.nan)
    ) * 100

    display(q_results.round(4))

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the specific action and state encoders used for Q-learning
action_le_q = joblib.load("action_le_q.pkl")
state_le_q = joblib.load("state_le_q.pkl")

# Load the processed validation dataframe
eval_df = pd.read_csv("dataco_rl_validation_processed.csv", encoding="latin1")

# Filter eval_df to only include state_ids that state_le_q knows about
known_original_state_ids = state_le_q.classes_

eval_df = eval_df[eval_df["state_id"].isin(known_original_state_ids)].copy()

if not eval_df.empty:
    # Transform eval_df's original state_id to Q-table's 0-indexed IDs
    eval_df["q_state_id"] = state_le_q.transform(eval_df["state_id"].astype(int))

    # Extract best action for each q_state_id from the Q-table
    # Q is assumed to be available in the global scope
    policy_action_ids = np.argmax(Q, axis=1)

    # Inverse transform to get the human-readable recommended action strings
    recommended_action_strings = action_le_q.inverse_transform(policy_action_ids).astype(str)

    # Create a temporary DataFrame to map q_state_id to recommended_action
    policy_df_local = pd.DataFrame({
        "q_state_id": np.arange(len(recommended_action_strings)),
        "recommended_action": recommended_action_strings
    })

    # Merge recommended actions into eval_df
    eval_df = eval_df.merge(policy_df_local, on="q_state_id", how="left")

    # Ensure the 'Shipping Mode_str' column is available
    if 'Shipping Mode_str' not in eval_df.columns:
        # Assuming 'Shipping Mode' column exists and needs to be copied
        if 'Shipping Mode' in eval_df.columns:
            eval_df['Shipping Mode_str'] = eval_df['Shipping Mode']
        else:
            print("Warning: 'Shipping Mode' column not found, cannot create 'Shipping Mode_str'.")

    # Now perform the comparison using the newly added 'recommended_action' column
    comparison = pd.crosstab(
        eval_df["Shipping Mode_str"],
        eval_df["recommended_action"].str.split("|").str[-1].str.strip(),
        normalize="index"
    ) * 100

    display(comparison.round(2))
else:
    print("Eval DataFrame is empty after filtering for Q-table compatible states. Cannot perform comparison.")

In [ ]:
import joblib
import pandas as pd # Import pandas (added for consistency)

# Load the action_le_q and state_le_q for correct transformation
action_le_q = joblib.load("action_le_q.pkl")
state_le_q = joblib.load("state_le_q.pkl") # Load state_le_q

# Load the processed eval_df
eval_df = pd.read_csv("dataco_rl_validation_processed.csv", encoding="latin1")

# Map eval_df's existing state_id (from global state_encoder) to Q-table's 0-indexed IDs
# The state_le_q was fitted on the integer IDs output by the global state_encoder.
# So, transforming eval_df's state_id with state_le_q will map them to the correct 0-indexed range.

# Filter eval_df to only include state_ids that state_le_q knows about
# This prevents errors if eval_df contains state_ids not seen during Q-table fitting.
known_original_state_ids = state_le_q.classes_

# Make a copy to avoid SettingWithCopyWarning and ensure q_state_id is properly set
if 'state_id' in eval_df.columns:
    eval_df_temp = eval_df[eval_df["state_id"].isin(known_original_state_ids)].copy()
    if not eval_df_temp.empty:
        eval_df_temp["q_state_id"] = state_le_q.transform(eval_df_temp["state_id"].astype(int))
    else:
        eval_df_temp["q_state_id"] = [] # Ensure column exists even if empty
    eval_df = eval_df_temp
else:
    print("Warning: 'state_id' column not found in eval_df. Ensure eval_df is properly prepared.")
    if eval_df.empty:
        eval_df["q_state_id"] = []

# Before getting recommended action, ensure eval_df is not empty
if not eval_df.empty:
    # Before populating recommended_action, drop any existing column to prevent conflicts on re-run
    if 'recommended_action' in eval_df.columns:
        eval_df = eval_df.drop(columns=['recommended_action'])

    # Use q_state_id to get recommended action from Q table
    eval_df["recommended_action_id"] = eval_df["q_state_id"].map(lambda s: np.argmax(Q[s]))

    # Inverse transform using action_le_q, not the global action_encoder
    eval_df["recommended_action"] = action_le_q.inverse_transform(eval_df["recommended_action_id"]).astype(str) # Ensure string type
else:
    print("Eval DataFrame is empty. Cannot determine recommended actions.")
    # Initialize empty columns to avoid KeyErrors in subsequent steps
    eval_df["recommended_action_id"] = []
    eval_df["recommended_action"] = []


# Estimate outcomes by historical averages for each recommended action
# Ensure train_df is available from the filtered data (from i49TEKAz3hsf)
action_outcomes = train_df.groupby("action")[[ # train_df here refers to the filtered one from i49TEKAz3hsf
    "Order Profit Per Order",
    "Days for shipping (real)"
]].mean()

# Merge only if eval_df is not empty and recommended_action is populated
if not eval_df.empty and 'recommended_action' in eval_df.columns and not eval_df['recommended_action'].isnull().all():
    eval_df = eval_df.merge(
        action_outcomes,
        left_on="recommended_action",
        right_index=True,
        suffixes=('_historical', '_policy')
    )

    historical_profit = eval_df["Order Profit Per Order_historical"].mean()
    policy_profit = eval_df["Order Profit Per Order_policy"].mean()

    historical_days = eval_df["Days for shipping (real)_historical"].mean()
    policy_days = eval_df["Days for shipping (real)_policy"].mean()

    profit_change_pct = ((policy_profit - historical_profit) / historical_profit) * 100
    days_change_pct = ((historical_days - policy_days) / historical_days) * 100

    print(f"Historical Profit: {historical_profit:.4f}")
    print(f"Policy Profit: {policy_profit:.4f}")
    print(f"Profit Change %: {profit_change_pct:.2f}%")

    print(f"Historical Days: {historical_days:.4f}")
    print(f"Policy Days: {policy_days:.4f}")
    print(f"Delivery Time Reduction %: {days_change_pct:.2f}%")

else:
    print("Eval DataFrame is empty or recommended_action is not available for outcome estimation.")
    # Initialize these variables to avoid NameError in MH8EThACvaGO
    historical_profit = 0
    policy_profit = 0
    profit_change_pct = 0
    historical_days = 0
    policy_days = 0
    days_change_pct = 0

In [ ]:
profit_goal = profit_change_pct > 0
delivery_goal = days_change_pct > 0

if profit_goal and delivery_goal:
    print("Goal achieved")
else:
    print("Goal not achieved")

In [ ]:
summary = train_df.groupby("action").agg(
    Profit=("Order Profit Per Order","mean"),
    Days=("Days for shipping (real)","mean"),
    Orders=("action","size")
)

summary["ProfitGain%"] = (
    (summary["Profit"] - summary["Profit"].mean())
    / summary["Profit"].mean()
) * 100

summary["DayReduction%"] = (
    (summary["Days"].mean() - summary["Days"])
    / summary["Days"].mean()
) * 100

summary.sort_values(
    ["ProfitGain%","DayReduction%"],
    ascending=False
).head(30)

In [ ]:
recommended_summary = (
    eval_df.groupby("recommended_action")
    .agg(
        Orders=("recommended_action", "size"),
        Profit=("Order Profit Per Order_policy", "mean"),
        Days=("Days for shipping (real)_policy", "mean")
    )
    .sort_values("Orders", ascending=False)
)

print(recommended_summary.head(20))

# Save Results

In [ ]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")